# 08 — 批处理模式与多设备同步

本笔记本介绍协调多台 Ivium 设备同时运行的高级模式：

- **状态参数** — Python 与 IviumSoft 共享的全局整数，用作同步信号
- **多设备设定点** — 一次调用即可向多台设备施加电位或电流，无需切换实例
- **并行实验模式** — 在所有设备上同时启动并监控测量

### 典型使用场景

- 触发所有设备同时开始测量
- 让 IviumSoft 端脚本等待 Python 信号后再继续
- 同时向多台设备施加相同设定点
- 并行运行独立实验并收集所有结果

### 前提条件
- 多个 IviumSoft 实例运行中（或一个带多通道的 Ivium-n-Soft）
- 多设备部分至少连接两台设备
- 错误处理参考：`01_getting_started.ipynb` — 第 4 节

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from pyvium import Pyvium

Pyvium.open_driver()
print(f"活跃实例: {Pyvium.get_active_iviumsoft_instances()}")

## 多通道 vs MC 模式 — 选择合适的架构

IviumSoft 提供两种不同的多设备模式。根据通道数选择：

| 特性 | **多通道** | **MC 模式** |
|---------|-----------------|-------------|
| 最大通道数 | **32** | **128** |
| 实时显示 | 是 — 实时电流/电位波形 | 否 |
| 运行期间数据访问 | `get_available_data_points_number()` 有效 | 完成前不可访问 |
| IviumSoft 控制 | `select_channel(n)`（在一个窗口内） | 独立的 IviumSoft 实例或脚本控制 |
| 使用场景 | ≤32 电极，需要实时监控 | >32 电极，更注重吞吐量 |
| Pyvium API | 每通道 `select_channel(n)` | 每实例 `select_iviumsoft_instance(n)` |

> **经验法则：** ≤32 通道使用**多通道**（Ivium-n-Soft）。>32 通道使用**MC 模式**配合多个 IviumSoft 实例。

> **注意：** 多通道模式使用 `select_channel(n)` 在**单个** IviumSoft 窗口内切换通道。MC 模式使用 `select_iviumsoft_instance(n)` 访问独立的 IviumSoft 窗口。

## 1. 状态参数

`set_status_par(value)` / `get_status_par()` 公开了 Python 与 IviumSoft 之间共享的全局整数。它是每实例的 — 每个 IviumSoft 窗口有自己的状态参数。

**来自 IviumSoft 手册的关键事实：**
- StatusPar **在 PC 启动时初始化为 `-1`**
- 它**保留上次设置的值**直到 PC 重启 — 不会在测量或驱动打开/关闭之间重置
- Python（通过 pyvium）和 IviumSoft 批处理脚本均可读写它

### 同步约定

选择适合你工作流的约定。常见约定：

| 值 | 设置方 | 含义 |
|-------|------------|------|
| `-1` | PC 启动默认 | 未初始化 |
| `0` | Python | 重置 / 空闲 |
| `1` | Python |「就绪 — IviumSoft 可以开始」 |
| `2` | IviumSoft 批处理 | 「测量运行中」 |
| `3` | IviumSoft 批处理 | 「测量完成」 |
| `-99` | 任一方 | 中止 / 错误 |

In [ ]:
Pyvium.select_iviumsoft_instance(1)

# 写入值
Pyvium.set_status_par(0)
print(f"状态参数设置为 0")

# 读回
val = Pyvium.get_status_par()
print(f"获取状态参数: {val}")

## 2. Python → IviumSoft 触发

模式：Python 将状态参数设置为 `1` 以告知 IviumSoft 开始方法。IviumSoft 端脚本轮询状态参数，看到 `1` 时开始执行。

这使 IviumSoft 端自动化能够响应 Python 事件。

In [ ]:
# 告知 IviumSoft 继续
Pyvium.select_iviumsoft_instance(1)
Pyvium.set_status_par(1)  # "执行"
print("已触发 IviumSoft 实例 1（状态参数 = 1）")

## 3. 完整双向协议

状态参数**双向**工作。IviumSoft 批处理脚本有两个对应的 DirectCommand：

| 方向 | Python 端 | IviumSoft 批处理端 |
|-----------|-------------|----------------------|
| Python → IviumSoft | `set_status_par(1)` | `WaitForStatusPar` — 批处理暂停直到值匹配 |
| IviumSoft → Python | 轮询循环中的 `get_status_par()` | `SetStatusPar` — 批处理完成时设置值 |

### IviumSoft 批处理脚本配置（供参考）

要使 IviumSoft 批处理脚本等待 Python 的启动信号并在完成后报告，请在批处理编辑器中配置 `DirectCommand` 行：

```
DirectCommand:
  WaitForStatusPar  ✓  Value = 1   Timeout = 300 s   ← 等待 Python 说「执行」
  （然后运行你的方法）
  SetStatusPar      ✓  Value = 3                      ← 向 Python 发出「完成」信号
```

Python 驱动序列：

```python
Pyvium.set_status_par(0)   # 重置
# ... 准备实验 ...
Pyvium.set_status_par(1)   # 告知 IviumSoft 开始
# ... 等待 IviumSoft 将状态设为 3 ...
```

In [ ]:
def wait_for_status(instance: int, target_value: int, timeout_s: float = 120.0):
    """阻塞直到指定实例的状态参数达到 target_value，或超时。"""
    Pyvium.select_iviumsoft_instance(instance)
    t_start = time.time()

    while True:
        current = Pyvium.get_status_par()
        if current == target_value:
            return True
        if time.time() - t_start > timeout_s:
            print(f"等待实例 {instance} 状态={target_value} 超时")
            return False
        time.sleep(0.2)

# 示例：等待最多 60 s 直到 IviumSoft 将状态参数设为 3（「完成」）
# done = wait_for_status(instance=1, target_value=3, timeout_s=60.0)
# if done:
#     print("测量完成")
print("wait_for_status() 辅助函数已定义")

## 4. 替代触发器 — 数字输出和模拟输入

除 StatusPar 外，IviumSoft 批处理模式还支持两种 Python 可直接驱动的硬件级触发机制：

### 数字触发器（`WaitForDigIn1`）

IviumSoft 批处理可在配置为以下内容的 `DirectCommand` 行处暂停：
```
WaitForDigIn1  ✓   WaitForHi = ✓   Timeout = 60 s
```
Python 通过在外部端口上置位或释放数字输出线来驱动它，该线接回设备的数字输入 1。

### 模拟触发器（`WaitForAn1`）

IviumSoft 批处理可暂停直到模拟输入 1 超过阈值电压：
```
WaitForAn1  ✓   UntilAn1 > 2.5 V   Timeout = 60 s
```
Python 通过将 DAC 通道 0 设置为触发电压来驱动它。

两种机制都有**超时** — 一旦超时，批处理脚本无论如何都会继续，因此实验不会因 Python 信号失败而永久停滞。

In [ ]:
import time

# --- 数字触发器模式 ---
# IviumSoft 批处理配置了 WaitForDigIn1（WaitForHi，Timeout=60s）
# Python 置位数字输出第 0 线以释放批处理暂停

def trigger_batch_via_digital(line: int = 0):
    """置位数字输出线为高电平以触发 IviumSoft 批处理 WaitForDigIn。"""
    Pyvium.set_digital_output(1 << line)   # 驱动该线为高电平
    print(f"数字第 {line} 线为高电平 — IviumSoft 批处理触发信号已发送")

def release_digital_trigger(line: int = 0):
    """批处理收到触发后释放数字线。"""
    current = Pyvium.get_digital_input()
    Pyvium.set_digital_output(0)            # 所有线为低电平
    print("数字线已重置为低电平")

# 示例用法：
# trigger_batch_via_digital(0)
# time.sleep(0.1)
# release_digital_trigger(0)

print("数字触发器辅助函数已定义")

# --- 模拟触发器模式 ---
# IviumSoft 批处理配置了 WaitForAn1（UntilAn1 > 2.5 V，Timeout=60s）
# Python 将 DAC 0 升至阈值以上以释放暂停

TRIGGER_VOLTAGE  = 3.0   # V — 必须超过批处理阈值
IDLE_VOLTAGE     = 0.0   # V

def trigger_batch_via_analog():
    Pyvium.set_dac(0, TRIGGER_VOLTAGE)
    print(f"DAC 0 → {TRIGGER_VOLTAGE} V — IviumSoft 批处理模拟触发信号已发送")

def release_analog_trigger():
    Pyvium.set_dac(0, IDLE_VOLTAGE)
    print(f"DAC 0 → {IDLE_VOLTAGE} V（空闲）")

# 示例用法：
# trigger_batch_via_analog()
# time.sleep(0.5)
# release_analog_trigger()

print("模拟触发器辅助函数已定义")

## 5. 从 IviumSoft 批处理脚本启动 Python（`ExecuteProgram`）

IviumSoft 批处理模式有一个 `ExecuteProgram` DirectCommand，可在批处理序列的特定位置启动任何外部可执行文件（包括 Python 脚本）。配置 `WaitUntilFinished = ✓` 时，批处理暂停直到 Python 进程退出。

**批处理配置：**
```
DirectCommand:
  ExecuteProgram     ✓
  Program Name:      python.exe
  Path:              C:\Users\...\Scripts\
  Command Line:      C:\path\to\my_analysis.py --output C:\data\result.csv
  WaitUntilFinished: ✓
```

**使用场景：**
- 在批处理循环中每次扫描后立即后处理数据
- 长时间实验完成后发送通知（邮件、Slack）
- 读取测量结果并写入下一个批处理行使用的新参数文件

**从 IviumSoft 角度的流程：**
```
[批处理第 N 行]  ExecuteMethod           ← 运行实验
[批处理第 N+1 行] ExecuteProgram         ← 启动 Python，等待退出
[批处理第 N+2 行] 循环返回 / 下一次扫描  ← Python 结果已在磁盘上
```

这是上述模式的反向：不是 Python 驱动 IviumSoft，而是 IviumSoft 在序列的恰当时刻驱动 Python。

## IviumSoft 批处理 — EditMethod 与 SI 单位

当 IviumSoft 批处理脚本使用 **`EditMethod`** DirectCommand 修改方法参数时，所有参数值必须以 **SI 基本单位**指定 — 无论 IviumSoft UI 中显示的是什么显示单位。

当 Python 生成随后由 IviumSoft 批处理脚本读取的参数值时（例如写入批处理读取的文件），或者在调试批处理修改的方法行为不符合预期时，这一点尤为重要。

| 参数 | IviumSoft 显示单位 | EditMethod / Pyvium 的 SI 单位 |
|-----------|--------------------------|--------------------------------|
| 电位 | mV | **V** |
| 电流 | µA、mA | **A** |
| 时间 | ms、min、h | **s** |
| 频率 | kHz | **Hz** |
| 扫速 | mV/s | **V/s** |

> **这与 Pyvium 的 `set_method_parameter()` 使用相同的约定** — 所有值均为 SI 基本单位。IviumSoft 在内部处理单位显示转换。

> **陷阱：** 如果将 `'scanrate'` 设置为 `'50'` 期望 50 mV/s，实际扫速将为 50 V/s。50 mV/s 应使用 `'0.05'`。

## 4. 同时触发所有实例

通过快速循环所有实例并设置状态参数，可以近乎同时触发所有设备。

In [ ]:
def broadcast_status(value: int):
    """在所有活跃 IviumSoft 实例上设置状态参数。"""
    instances = Pyvium.get_active_iviumsoft_instances()
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        Pyvium.set_status_par(value)
    print(f"已向实例 {instances} 广播状态={value}")

broadcast_status(0)   # 将所有重置为空闲
time.sleep(1.0)
broadcast_status(1)   # 触发所有

## 5. 多设备设定点

`set_device_potential(instance, V)` 和 `set_device_current(instance, A)` 向特定设备实例施加设定点，**无需更改当前选中的实例**。这允许在不切换实例开销的情况下快速更新所有设备的设定点。

- `instance`：IviumSoft 实例编号（从 1 开始）
- 两个函数都需要驱动已打开，但**不需要**目标设备是当前选中的实例

In [ ]:
# 同时向所有活跃实例施加相同电位
HOLD_POTENTIAL = 0.1  # V

instances = Pyvium.get_active_iviumsoft_instances()
for inst in instances:
    Pyvium.set_device_potential(inst, HOLD_POTENTIAL)
    print(f"实例 {inst}: 电位 → {HOLD_POTENTIAL} V")

print("所有设备已设置")

In [ ]:
# 跨设备施加电位梯度
potentials = [0.0, 0.1, 0.2, 0.3]  # V — 每实例一个

for inst, V in zip(instances, potentials):
    Pyvium.set_device_potential(inst, V)
    print(f"实例 {inst}: {V:.2f} V")

## 6. 并行在所有设备上运行方法

在每台设备上启动方法，然后在所有设备完成后收集数据。

In [ ]:
METHOD_PATH     = r"C:/IviumStat/datafiles/TEST1.imf"
EXPECTED_POINTS = 160
POLL_INTERVAL   = 0.5

def start_all(instances, method_path):
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        Pyvium.connect_device()
        Pyvium.start_method(method_path)
        print(f"实例 {inst}: 方法已启动")

def wait_all(instances, expected_points):
    remaining = set(instances)
    while remaining:
        finished = set()
        for inst in remaining:
            Pyvium.select_iviumsoft_instance(inst)
            n = Pyvium.get_available_data_points_number()
            if n >= expected_points:
                finished.add(inst)
        for inst in finished:
            print(f"  实例 {inst}: 完成")
        remaining -= finished
        if remaining:
            time.sleep(POLL_INTERVAL)
    print("所有测量完成")

def collect_all(instances):
    results = {}
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        n = Pyvium.get_available_data_points_number()
        rows = [Pyvium.get_data_point(i) for i in range(1, n + 1)]
        results[inst] = pd.DataFrame(rows, columns=["E (V)", "I (A)", "aux"])
        print(f"  实例 {inst}: 已收集 {len(rows)} 个点")
    return results

print("辅助函数已定义。取消下方注释以运行:")
# start_all(instances, METHOD_PATH)
# wait_all(instances, EXPECTED_POINTS)
# data = collect_all(instances)

## 7. 绘制多设备结果

In [ ]:
# 假设 `data` 字典已由上方 collect_all() 填充
# data = collect_all(instances)

# 用于演示的合成示例
import numpy as np
data = {
    1: pd.DataFrame({"E (V)": np.linspace(0, 1, 100),
                     "I (A)": np.random.randn(100) * 1e-6 + 1e-5}),
    2: pd.DataFrame({"E (V)": np.linspace(0, 1, 100),
                     "I (A)": np.random.randn(100) * 1e-6 + 2e-5}),
}

fig, ax = plt.subplots(figsize=(8, 5))
for inst, df in data.items():
    ax.plot(df["E (V)"], df["I (A)"] * 1e6, label=f"设备 {inst}")

ax.set_xlabel("电位 (V)")
ax.set_ylabel("电流 (µA)")
ax.set_title("并行 LSV — 所有设备")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. 完整同步工作流

完整模板：连接所有设备，并行运行方法，收集数据，保存并绘图。

## 同步通道 — 主通道最后启动规则

在同步模式下运行多个通道时（使用 IviumSoft 的通道同步功能），**主通道必须始终最后启动**。主通道充当所有同步通道的时钟源；从通道在主通道启动前处于等待就绪状态。

**关键规则：** 如果先启动主通道，同步信号会在任何从通道监听之前触发 — 所有从通道都会错过触发信号并独立启动（不同步）。

```python
# 同步多通道启动的正确顺序
slaves  = [2, 3, 4]  # 从通道实例
master  = 1          # 主通道实例

# 1. 先启动所有从通道（它们进入等待状态）
for inst in slaves:
    Pyvium.select_iviumsoft_instance(inst)
    Pyvium.start_method(method_path)
    print(f"从实例 {inst}: 等待同步")

# 2. 短暂延迟确保从通道已就绪
time.sleep(0.5)

# 3. 最后启动主通道 — 触发同步信号
Pyvium.select_iviumsoft_instance(master)
Pyvium.start_method(method_path)
print(f"主实例 {master}: 已启动 — 同步触发信号已发送")
```

> 此规则适用于 IviumSoft 的同步通道功能（在多通道设置中配置）。与上述各节中描述的软件级状态参数同步不同，后者对时序要求不严格。

In [ ]:
import os

METHOD_PATH    = r"C:/IviumStat/datafiles/TEST1.imf"
OUTPUT_DIR     = r"C:/IviumStat/Data"
EXPECTED_PTS   = 160

Pyvium.open_driver()
instances = Pyvium.get_active_iviumsoft_instances()
print(f"设备: {instances}")

# 第 1 步：全部连接并启动
for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    Pyvium.connect_device()

for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    Pyvium.start_method(METHOD_PATH)
    print(f"  实例 {inst}: 已启动")

# 第 2 步：等待所有完成
remaining = set(instances)
while remaining:
    done = set()
    for inst in remaining:
        Pyvium.select_iviumsoft_instance(inst)
        if Pyvium.get_available_data_points_number() >= EXPECTED_PTS:
            done.add(inst)
    remaining -= done
    if remaining:
        time.sleep(0.5)
print("全部完成")

# 第 3 步：收集并保存
for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    out = os.path.join(OUTPUT_DIR, f"device_{inst}.idf")
    if os.path.isdir(OUTPUT_DIR):
        Pyvium.save_data(out)
        print(f"  实例 {inst} → {out}")

Pyvium.close_driver()

---

## 小结

### 状态参数（同步）

| 任务 | Python | IviumSoft 批处理端 |
|------|--------|----------------------|
| 写入状态 | `set_status_par(value)` | `SetStatusPar` DirectCommand |
| 读取/等待状态 | 轮询循环中的 `get_status_par()` | `WaitForStatusPar` DirectCommand |
| 广播到所有实例 | 循环 + `select_iviumsoft_instance(n)` + `set_status_par(v)` | — |
| PC 启动时的初始值 | — | **-1**（保留直到 PC 重启） |

### 替代硬件触发器

| 触发类型 | Python 发送 | IviumSoft 批处理等待 |
|-------------|-------------|------------------------|
| 数字 | `set_digital_output(bitmask)` | `WaitForDigIn1`（高/低，带超时） |
| 模拟 | `set_dac(0, voltage)` | `WaitForAn1 > threshold`（带超时） |
| 启动 Python | — | `ExecuteProgram`（可选 `WaitUntilFinished`） |

### 多设备设定点（无需切换实例）

| 任务 | 方法 |
|------|--------|
| 在任意实例上设置电位 | `set_device_potential(instance, V)` |
| 在任意实例上设置电流 | `set_device_current(instance, A)` |

### 多通道架构选择

| 通道数 | 架构 | Pyvium API |
|----------|-------------|-------------|
| ≤ 32 | 多通道（Ivium-n-Soft）— 实时显示 | `select_channel(n)` |
| > 32 | MC 模式 — 无实时显示 | `select_iviumsoft_instance(n)` |

### 同步通道规则

**始终在主通道之前启动从通道。** 主通道在启动时触发同步信号 — 从通道必须已处于等待状态。

### EditMethod / set_method_parameter 单位

所有参数值均使用 **SI 基本单位**（V、A、s、Hz），无论 IviumSoft UI 中显示什么单位。

### 并行测量模式

```python
# 1. 全部启动
for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    Pyvium.start_method(path)

# 2. 等待所有完成
while not all_done(instances):
    time.sleep(0.5)

# 3. 从所有设备收集
for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    data[inst] = read_all_points()
```

---

## 笔记本系列完毕

| 笔记本 | 主题 | 是否需要硬件？ |
|----------|-------|:-:|
| 01 | 入门、错误处理 | 否（仅需 IviumSoft） |
| 02 | 设备与实例管理 | 是 |
| 03 | 直接模式 — 恒电位仪基础 | 是 |
| 04 | 直接模式 — 波形、DAC/ADC、数字 I/O、AC | 是 |
| 05 | BiStat（WE2）和 WE32 多通道 | 是（+ 附件） |
| 06 | 方法模式 — 运行实验、读取数据 | 是 |
| 07 | IDF 文件解析和 CSV 导出 | **否** |
| 08 | 批处理模式与多设备同步 | 是（多台设备） |